Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\24pasos_mlp_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52381, 72)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 72)
Las dimensiones de testX son:  (10529, 72)
Las dimensiones de valX son:  (5186, 72)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 4s - 16ms/step - ia: 0.2425 - loss: 4.5528 - mae: 1.5442 - rmse: 2.0369 - smape: 1.4953 - val_ia: 0.2119 - val_loss: 1.2896 - val_mae: 0.9321 - val_rmse: 1.0088 - val_smape: 1.6700

Epoch 2/128                                           

231/231 - 0s - 2ms/step - ia: 0.2986 - loss: 1.7288 - mae: 1.0013 - rmse: 1.2964 - smape: 1.4258 - val_ia: 0.2372 - val_loss: 0.9500 - val_mae: 0.8045 - val_rmse: 0.8752 - val_smape: 1.5557

Epoch 3/128                                           

231/231 - 1s - 3ms/step - ia: 0.3126 - loss: 1.2476 - mae: 0.8632 - rmse: 1.1032 - smape: 1.4167 - val_ia: 0.2435 - val_loss: 0.8688 - val_mae: 0.7759 - val_rmse: 0.8418 - val_smape: 1.5120

Epoch 4/128                                           

231/231 - 1s - 2ms/step - ia: 0.3205 - loss: 1.0399 - mae: 0.8028 - rmse: 1.0087 - smape: 1.4217 - val_ia: 0.2477 - val_loss: 0.7949 - val_mae: 0.7462 - val_rmse: 0.8096 - val_smape: 1.4566

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



29/29 - 3s - 99ms/step - ia: 0.5572 - loss: 0.4175 - mae: 0.5006 - rmse: 0.6256 - smape: 0.9647 - val_ia: 0.6800 - val_loss: 0.2358 - val_mae: 0.3971 - val_rmse: 0.4731 - val_smape: 0.7316

Epoch 2/16                                                                       

29/29 - 0s - 8ms/step - ia: 0.8004 - loss: 0.1450 - mae: 0.2873 - rmse: 0.3788 - smape: 0.5629 - val_ia: 0.7093 - val_loss: 0.1821 - val_mae: 0.3372 - val_rmse: 0.4188 - val_smape: 0.6958

Epoch 3/16                                                                       

29/29 - 0s - 6ms/step - ia: 0.8429 - loss: 0.0922 - mae: 0.2284 - rmse: 0.3030 - smape: 0.4944 - val_ia: 0.7333 - val_loss: 0.1576 - val_mae: 0.3083 - val_rmse: 0.3887 - val_smape: 0.6473

Epoch 4/16                                                                       

29/29 - 0s - 5ms/step - ia: 0.8649 - loss: 0.0698 - mae: 0.1972 - rmse: 0.2634 - smape: 0.4552 - val_ia: 0.7498 - val_loss: 0.1493 - val_mae: 0.3019 - val_rmse: 0.3794 - val_smape: 0.

...

461/461 - 4s - 8ms/step - ia: 0.4140 - loss: 0.8692 - mae: 0.7437 - rmse: 0.9132 - smape: 1.2480 - val_ia: 0.2244 - val_loss: 0.4362 - val_mae: 0.5362 - val_rmse: 0.5745 - val_smape: 1.0183

Epoch 2/8                                                                         

461/461 - 1s - 2ms/step - ia: 0.5483 - loss: 0.5171 - mae: 0.5732 - rmse: 0.7088 - smape: 1.0269 - val_ia: 0.2648 - val_loss: 0.3995 - val_mae: 0.5029 - val_rmse: 0.5424 - val_smape: 0.9945

Epoch 3/8                                                                         

461/461 - 1s - 2ms/step - ia: 0.6084 - loss: 0.3940 - mae: 0.5021 - rmse: 0.6182 - smape: 0.9345 - val_ia: 0.2816 - val_loss: 0.3435 - val_mae: 0.4619 - val_rmse: 0.4987 - val_smape: 0.9621

Epoch 4/8                                                                         

461/461 - 1s - 2ms/step - ia: 0.6369 - loss: 0.3411 - mae: 0.4627 - rmse: 0.5737 - smape: 0.8781 - val_ia: 0.2916 - val_loss: 0.3081 - val_mae: 0.4392 - val_rmse: 0.4734 - val

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



116/116 - 3s - 22ms/step - ia: 0.4589 - loss: 0.7083 - mae: 0.6627 - rmse: 0.8258 - smape: 1.1777 - val_ia: 0.4954 - val_loss: 0.3871 - val_mae: 0.5274 - val_rmse: 0.5957 - val_smape: 0.8767

Epoch 2/16                                                                        

116/116 - 0s - 2ms/step - ia: 0.6250 - loss: 0.3923 - mae: 0.4890 - rmse: 0.6222 - smape: 0.8653 - val_ia: 0.4670 - val_loss: 0.4095 - val_mae: 0.5443 - val_rmse: 0.6154 - val_smape: 0.9494

Epoch 3/16                                                                        

116/116 - 0s - 2ms/step - ia: 0.6602 - loss: 0.3406 - mae: 0.4583 - rmse: 0.5796 - smape: 0.7998 - val_ia: 0.5202 - val_loss: 0.3014 - val_mae: 0.4594 - val_rmse: 0.5273 - val_smape: 0.8536

Epoch 4/16                                                                        

116/116 - 0s - 2ms/step - ia: 0.6793 - loss: 0.3066 - mae: 0.4359 - rmse: 0.5531 - smape: 0.7799 - val_ia: 0.4616 - val_loss: 0.4082 - val_mae: 0.5491 - val_rmse: 0.6177 - va

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



58/58 - 2s - 28ms/step - ia: 0.3715 - loss: 0.7705 - mae: 0.7084 - rmse: 0.8664 - smape: 1.3486 - val_ia: 0.5383 - val_loss: 0.4682 - val_mae: 0.5705 - val_rmse: 0.6600 - val_smape: 1.0834

Epoch 2/256                                                                       

58/58 - 0s - 2ms/step - ia: 0.4915 - loss: 0.5315 - mae: 0.5847 - rmse: 0.7275 - smape: 1.1500 - val_ia: 0.5700 - val_loss: 0.4241 - val_mae: 0.5344 - val_rmse: 0.6227 - val_smape: 1.0131

Epoch 3/256                                                                       

58/58 - 0s - 3ms/step - ia: 0.5364 - loss: 0.4851 - mae: 0.5495 - rmse: 0.6961 - smape: 1.0605 - val_ia: 0.5736 - val_loss: 0.4201 - val_mae: 0.5284 - val_rmse: 0.6183 - val_smape: 0.9922

Epoch 4/256                                                                       

58/58 - 0s - 2ms/step - ia: 0.5605 - loss: 0.4542 - mae: 0.5325 - rmse: 0.6717 - smape: 1.0167 - val_ia: 0.5852 - val_loss: 0.3971 - val_mae: 0.5097 - val_rmse: 0.6040 - val_smape:

In [16]:
print(best)

{'activation': 0, 'batch': 1, 'dropout': 0.0, 'epochs': 3, 'layers': 2.0, 'learning_rate': 0.006065391920254659, 'units': 1}


In [17]:
print(best)

{'activation': 0, 'batch': 1, 'dropout': 0.0, 'epochs': 3, 'layers': 2.0, 'learning_rate': 0.006065391920254659, 'units': 1}
